In [1]:
import os
import pandas as pd
import csv
from openpyxl import Workbook
import numpy as np
import time
import win32com.client as win32

FirstStepOfKill에서

for z in range(5)의 5 변경 가능

SecondStepOfKill에서

for i in range(5)의 5 변경 가능



FirstStepOfSortie에서

for z in range(5)의 5 변경 가능

SecondStepOfSortie에서

for i in range(5)의 5 변경 가능

In [6]:
def FirstStepOfKill():
    for z in range(5): # ◉◉◉◉바꿀 수 있는 숫자◉◉◉◉
        base_path = r"C:\Users\LAB1\Desktop\대기실" # ◉◉◉◉바꿀 수 있는 위치◉◉◉◉
        input_file = base_path + r"\kill{}.csv".format(z+1)
        output_file = base_path + r"\정리된kill{}.csv".format(z+1)

        with open(input_file, newline='', encoding='utf-8-sig') as f:
            reader = csv.reader(f)
            data = [row for row in reader]

        if data:
            header = data[3]
            header.extend(["Surface Target", "Number Killed"])
            data[3] = header

        num_cols = len(data[3]) if data else 0
        for i, row in enumerate(data):
            if len(row) < num_cols:
                data[i].extend([""] * (num_cols - len(row)))

        data2 = []
        for row in data:
            if row and isinstance(row[0], str) and "Surface Target" in row[0]:
                continue
            data2.append(row)

        numeric_indices = []
        numeric_values = []
        for idx, row in enumerate(data2):
            if idx >= 4:
                val = row[0].strip()
                if val.isdigit():
                    numeric_indices.append(idx)
                    numeric_values.append(int(val))
        sorted_events = sorted(zip(numeric_values, numeric_indices), key=lambda x: x[0])
        sorted_numeric_indices = [idx for _, idx in sorted_events]

        data3 = data2[:4].copy()
        for i, cur_idx in enumerate(sorted_numeric_indices):
            event_row = data2[cur_idx].copy()
            if len(event_row) < num_cols:
                event_row.extend([""] * (num_cols - len(event_row)))
            next_idx = None
            if i < len(sorted_numeric_indices) - 1:
                next_idx = sorted_numeric_indices[i+1]
            else:
                next_idx = len(data2)

            targets = []
            shots_missed = None
            for j in range(cur_idx + 1, next_idx):
                if j < len(data2):
                    row = data2[j]
                    if not row or len(row) == 0:
                        continue
                    first_cell = row[0].strip() if isinstance(row[0], str) else ""
                    if first_cell.startswith("Shot(s) Missed"):
                        shots_missed = ("Shot(s) Missed", row[1] if len(row) > 1 else "")
                    elif first_cell != "" and not first_cell.isdigit():
                        target_name = row[0]
                        number = row[1] if len(row) > 1 else ""
                        targets.append((target_name, number))
            if targets:
                first_target_name, first_number = targets[0]
                event_row[-2] = first_target_name
                event_row[-1] = first_number
                data3.append(event_row)
                for t_name, t_num in targets[1:]:
                    new_row = [""] * num_cols
                    new_row[-2] = t_name
                    new_row[-1] = t_num
                    data3.append(new_row)
                if shots_missed:
                    label, miss_num = shots_missed
                    new_row = [""] * num_cols
                    new_row[-2] = label
                    new_row[-1] = miss_num
                    data3.append(new_row)
            else:
                if shots_missed:
                    label, miss_num = shots_missed
                    event_row[-2] = label
                    event_row[-1] = miss_num
                data3.append(event_row)
                if shots_missed:
                    new_row = [""] * num_cols
                else:
                    pass

        data4 = []
        for row in data3:
            data4.append(row.copy())

        for idx in range(4, len(data4)):
            row = data4[idx]
            if row[0] == "":
                prev_row = data4[idx-1]
                if prev_row[0] != "":
                    for col in range(0, 11):
                        row[col] = prev_row[col]
                    data4[idx] = row

        header_section = data4[:4]
        data_section = data4[4:]

        def sort_key(row):
            try:
                return int(row[0])
            except:
                return float('inf')

        data_section.sort(key=sort_key)

        data_section = [row for row in data_section if any(cell.strip() for cell in row)]

        final_data = header_section + data_section

        with open(output_file, 'w', newline='', encoding='utf-8-sig') as f:
            writer = csv.writer(f)
            writer.writerows(final_data)
    
def SecondStepOfKill():
    folder_path = r"C:\Users\LAB1\Desktop\대기실" # ◉◉◉◉바꿀 수 있는 위치◉◉◉◉
    output_path = os.path.join(folder_path, f"합친 것_kill.csv")
    try:
        # 2. 결과를 저장할 '합친 것.csv' 파일을 쓰기 모드로 열기
        # (Excel에서 한글이 깨지지 않도록 utf-8-sig 인코딩 사용)
        with open(output_path, 'w', encoding='utf-8-sig', newline='') as outfile:

            excel_row = 2
            
            # 1부터 5까지 반복하며 파일 읽기
            for i in range(5): # ◉◉◉◉바꿀 수 있는 숫자◉◉◉◉
                file_name = f"정리된kill{i+1}.csv"
                file_path = os.path.join(folder_path, file_name)

                with open(file_path, 'r', encoding='utf-8-sig', newline='') as infile:
                    lines = infile.readlines()

                    if i == 0:
                        # [첫 번째 파일 - 정리된kill1.csv 처리]
                        # 1행 ~ 3행 (메타데이터): 열이 한 개 늘어났으므로 끝에 쉼표(,) 추가
                        for j in range(3):
                            outfile.write(lines[j].rstrip('\r\n') + ",\n")

                        # 4행 (데이터 헤더): N열 위치에 ',rep' 추가
                        outfile.write(lines[3].rstrip('\r\n') + ",Run_No" + ",Day" + ",SA235N" + ",isCNO\n")

                        # 5행부터 끝까지 (실제 데이터): 끝에 ',1' 추가
                        for line in lines[4:]:
                            if line.strip():  # 빈 줄 건너뛰기
                                #outfile.write(line.rstrip('\r\n') + f',{i+1},"=LEFT(K{excel_row},3)"\n')
                                outfile.write(line.rstrip('\r\n') + f',{i+1},"=LEFT(K{excel_row},3)","=IF(ISNA(VLOOKUP(G{excel_row},SA235!$B$1:$C$62,2,FALSE)),""ZEWGCI"",VLOOKUP(G{excel_row},SA235!$B$1:$C$62,2,FALSE))","=IF(ISNA(VLOOKUP(LEFT(G{excel_row},10),CNO!$B$2:$G$131,6,FALSE)),""not CNO"",VLOOKUP(LEFT(G{excel_row},10),CNO!$B$2:$G$131,6,FALSE))"\n')
                                excel_row += 1
                    else:
                        # [나머지 파일 - 정리된kill2~5.csv 처리]
                        # 1~4행은 버리고 5행(실제 데이터)부터 가져와서 끝에 ',2', ',3'... 추가
                        for line in lines[4:]:
                            if line.strip():
                                #outfile.write(line.rstrip('\r\n') + f',{i+1},"=LEFT(K{excel_row},3)"\n')
                                outfile.write(line.rstrip('\r\n') + f',{i+1},"=LEFT(K{excel_row},3)","=IF(ISNA(VLOOKUP(G{excel_row},SA235!$B$1:$C$62,2,FALSE)),""ZEWGCI"",VLOOKUP(G{excel_row},SA235!$B$1:$C$62,2,FALSE))","=IF(ISNA(VLOOKUP(LEFT(G{excel_row},10),CNO!$B$2:$G$131,6,FALSE)),""not CNO"",VLOOKUP(LEFT(G{excel_row},10),CNO!$B$2:$G$131,6,FALSE))"\n')
                                excel_row += 1

        print("✅ '합친 것.csv' 파일이 성공적으로 생성되었습니다!")
        print(f"저장 위치: {output_path}")

    except UnicodeDecodeError:
        print("❌ 인코딩 에러가 발생했습니다. 원본 파일 형식에 맞춰 코드의 'utf-8-sig'를 'cp949'로 모두 변경한 후 다시 실행해주세요.")
    except FileNotFoundError as e:
        print(f"❌ 파일을 찾을 수 없습니다. 경로에 파일이 모두 있는지 확인해주세요:\n{e}")
    
def FirstStepOfSortie(x):
    for z in range(5): # ◉◉◉◉바꿀 수 있는 숫자◉◉◉◉
        file_path = r"C:\Users\LAB1\Desktop\대기실\{}{}.csv".format(x, z+1) # ◉◉◉◉바꿀 수 있는 위치◉◉◉◉
        output_path = r"C:\Users\LAB1\Desktop\대기실\정리된{}{}.csv".format(x, z+1) # ◉◉◉◉바꿀 수 있는 위치◉◉◉◉

        with open(file_path, "r", encoding='utf-8') as f:
            max_cols = max(len(line.rstrip("\n").split(",")) for line in f)

        df = pd.read_csv(
            file_path,
            engine="python",
            header=None,
            names=list(range(max_cols)),
            encoding='utf-8',
        )

        grand_idx = df.index[df[0].astype(str).str.strip().eq("GrandTotal")]
        if len(grand_idx) > 0:
            df = df.iloc[: grand_idx[0], :].copy()

        #내가 손 본 곳 : try, except 제거 필요하다면...
        try:
            day_label_idx = df.index[df[0].astype(str).str.strip().eq("Day")][0]
            header_src_idx = df.index[
                (df[0].astype(str).str.strip().eq("Air Asset"))
                & (df[1].astype(str).str.strip().eq("Mission"))
            ][0]
        except:
            day_label_idx = df.index[df[0].astype(str).str.strip().eq("Day")][0]
            header_src_idx = df.index[
                (df[0].astype(str).str.strip().eq("Air Asset"))
                & (df[1].astype(str).str.strip().eq("Originating Location"))
            ][0]

        ncols = df.shape[1]

        header_row = [np.nan] * ncols
        header_row[0] = df.iat[day_label_idx, 0]
        header_row[1:] = df.iloc[header_src_idx, 0 : ncols - 1].tolist()

        def is_day_marker(val) -> bool:
            if pd.isna(val):
                return False
            s = str(val).strip()
            return s.isdigit() and len(s) <= 2

        data_rows = []
        current_day = None

        for idx in range(day_label_idx + 1, len(df) - 1):
            row = df.iloc[idx]

            v0 = row.iat[0]

            if is_day_marker(v0) and row.iloc[1:].isna().all():
                current_day = str(v0).strip()
                continue

            if str(v0).strip() == "Air Asset":
                continue

            if current_day is None:
                continue

            if pd.isna(row.iat[0]) or pd.isna(row.iat[1]):
                continue

            v1 = row.iat[1]
            if isinstance(v1, str) and v1.endswith("_S"):
                continue

            if pd.isna(pd.to_numeric(row.iat[2], errors="coerce")):
                continue

            new_row = [np.nan] * ncols
            new_row[0] = current_day
            new_row[1:] = row.iloc[0 : ncols - 1].tolist()
            data_rows.append(new_row)

        df_out = pd.concat(
            [
                df.iloc[0:3].reset_index(drop=True),
                pd.DataFrame([header_row], columns=df.columns),
                pd.DataFrame(data_rows, columns=df.columns),
            ],
            ignore_index=True,
        )

        csv_text = df_out.to_csv(index=False, header=False, line_terminator="\r\n")
        if csv_text.endswith("\r\n"):
            csv_text = csv_text[:-2]

        with open(output_path, "w", encoding='utf-8', newline="") as f:
            f.write(csv_text)
    
def SecondStepOfSortie(x):
    folder_path = r"C:\Users\LAB1\Desktop\대기실" # ◉◉◉◉바꿀 수 있는 위치◉◉◉◉
    #output_path = os.path.join(folder_path, "합친 것.csv")

    output_path = os.path.join(folder_path, f"합친 것_{x}.csv")
    try:
        # 2. 결과를 저장할 '합친 것.csv' 파일을 쓰기 모드로 열기
        # (Excel에서 한글이 깨지지 않도록 utf-8-sig 인코딩 사용)
        with open(output_path, 'w', encoding='utf-8-sig', newline='') as outfile:

            # 1부터 5까지 반복하며 파일 읽기
            for i in range(5): # ◉◉◉◉바꿀 수 있는 숫자◉◉◉◉
                file_name = f"정리된{x}{i+1}.csv"
                file_path = os.path.join(folder_path, file_name)

                with open(file_path, 'r', encoding='utf-8-sig', newline='') as infile:
                    lines = infile.readlines()

                    if i == 0:
                        # [첫 번째 파일 - 정리된kill1.csv 처리]
                        # 1행 ~ 3행 (메타데이터): 열이 한 개 늘어났으므로 끝에 쉼표(,) 추가
                        for j in range(3):
                            outfile.write(lines[j].rstrip('\r\n') + ",\n")

                        # 4행 (데이터 헤더): N열 위치에 ',rep' 추가
                        outfile.write(lines[3].rstrip('\r\n') + ",Run_No\n")

                        # 5행부터 끝까지 (실제 데이터): 끝에 ',1' 추가
                        for line in lines[4:]:
                            if line.strip():  # 빈 줄 건너뛰기
                                outfile.write(line.rstrip('\r\n') + f",{i+1}\n")
                    else:
                        # [나머지 파일 - 정리된kill2~5.csv 처리]
                        # 1~4행은 버리고 5행(실제 데이터)부터 가져와서 끝에 ',2', ',3'... 추가
                        for line in lines[4:]:
                            if line.strip():
                                outfile.write(line.rstrip('\r\n') + f",{i+1}\n")

        print("✅ '합친 것.csv' 파일이 성공적으로 생성되었습니다!")
        print(f"저장 위치: {output_path}")

    except UnicodeDecodeError:
        print("❌ 인코딩 에러가 발생했습니다. 원본 파일 형식에 맞춰 코드의 'utf-8-sig'를 'cp949'로 모두 변경한 후 다시 실행해주세요.")
    except FileNotFoundError as e:
        print(f"❌ 파일을 찾을 수 없습니다. 경로에 파일이 모두 있는지 확인해주세요:\n{e}")
        
def create_avg_pivot(prefix):
    pv_sheet_name = f"{prefix}_PV"
    avg_sheet_name = f"{prefix}_PV_AVG"
    raw_sheet_name = f"{prefix} 나열"
    
    try:
        ws_pv = wb_pv.Sheets(pv_sheet_name)
    except Exception:
        return
    
    ws_avg = wb_pv.Sheets.Add(After=ws_pv)
    ws_avg.Name = avg_sheet_name
    
    ws_pv.UsedRange.Copy()
    ws_avg.Range("A1").PasteSpecial(Paste=-4163)
    excel.Application.CutCopyMode = False
    
    try:
        ws_raw = wb_pv.Sheets(raw_sheet_name)
        m = excel.WorksheetFunction.Max(ws_raw.Columns(66))
        if m == 0 or m is None:
            m = excel.WorksheetFunction.Max(ws_raw.Columns(54))
            if m == 0 or m is None:
                m = 1
    except Exception:
        m = 1
        
    max_row = ws_avg.UsedRange.Rows.Count
    max_col = ws_avg.UsedRange.Columns.Count
    
    for r in range(3, max_row + 1):
        for c in range(4, max_col + 1):
            val = ws_avg.Cells(r, c).Value
            
            if type(val) in (int, float):
                ws_avg.Cells(r, c).Value = val / m
    
    print(f"{prefix}_PV_AVG 시트 생성")
    
def convert_value(value):
    value = value.strip()
    
    if value == "":
        return ""
    
    try:
        return int(value)
    except ValueError:
        pass
    
    try:
        return float(value)
    except ValueError:
        pass
    
    return value

def create_static_pivot(sheet_name, pv_sheet_name):
    try:
        ws = wb_pv.Sheets(sheet_name)
    except:
        return

    ws_pv = wb_pv.Sheets.Add(After=ws)
    ws_pv.Name = pv_sheet_name

    max_row = ws.UsedRange.Rows.Count
    max_col = ws.UsedRange.Columns.Count
    source_range = f"'{sheet_name}'!R1C1:R{max_row}C{max_col}"

    pc = wb_pv.PivotCaches().Create(SourceType=1, SourceData=source_range)
    pt = pc.CreatePivotTable(TableDestination=f"'{pv_sheet_name}'!R3C1", TableName=f"PT_{pv_sheet_name}")

    row_fields = ["Day", "Air Asset", "Mission", "Originating Location"]
    for idx, field_name in enumerate(row_fields):
        try:
            pf = pt.PivotFields(field_name)
            pf.Orientation = 1
            pf.Position = idx + 1

            pf.Subtotals = (False, False, False, False, False, False, False, False, False, False, False, False)

            pf.RepeatLabels = True
        except:
            pass
    pt.RowAxisLayout(1)

    for col_idx in range(4, 66):
        val_header = ws.Cells(1, col_idx).Value
        if val_header:
            try:
                data_field = pt.AddDataField(pt.PivotFields(val_header), f"합계 : {val_header}")
                #data_field.Function = -4112
            except Exception as e:
                pass

    print(f"{pv_sheet_name} 시트 생성")

def create_dynamic_pivot():
    sheet_name = 'A2S kill 나열'
    pv_sheet_name = 'A2S_PV'
    try:
        ws = wb_pv.Sheets(sheet_name)
    except:
        return

    ws_pv = wb_pv.Sheets.Add(After=ws)
    ws_pv.Name = pv_sheet_name

    max_row = ws.UsedRange.Rows.Count
    max_col = ws.UsedRange.Columns.Count
    source_range = f"'{sheet_name}'!R1C1:R{max_row}C{max_col}"

    pc = wb_pv.PivotCaches().Create(SourceType=1, SourceData=source_range)
    pt = pc.CreatePivotTable(TableDestination=f"'{pv_sheet_name}'!R3C1", TableName="PT_A2S")

    for field_name in ["Victim Site Type", "Surface Target"]:
        try:
            pf_filter = pt.PivotFields(field_name)
            pf_filter.Orientation = 3
        except:
            pass

    try:
        pf_col = pt.PivotFields("Day")
        pf_col.Orientation = 2
    except:
        pass

    try:
        pf_row = pt.PivotFields("Victim")
        pf_row.Orientation = 1
    except:
        pass

    try:
        a2s_field = pt.AddDataField(pt.PivotFields("Number Killed"), "합계 : Number Killed")
        #a2s_field.Function = -4112
    except:
        pass

    print(f"A2S kill Pivot 시트 생성")
    
def create_final_tab(prefix, ko_name):
    avg_sheet_name = f"{prefix}_PV_AVG"
    final_sheet_name = f"{ko_name}_최종"

    try:
        ws_avg = wb_pv.Sheets(avg_sheet_name)
    except Exception:
        return

    ws_final = wb_pv.Sheets.Add(After=ws_avg)
    ws_final.Name = final_sheet_name

    ws_final.Range("A1").Value = f"Daily Sorties by Air Asset and Mission {prefix}"
    ws_final.Range("A1").Font.Name = "맑은 고딕"
    ws_final.Range("A1").Font.Size = 16
    ws_final.Range("A1").Font.Bold = True

    max_row = ws_avg.UsedRange.Rows.Count
    max_col = ws_avg.UsedRange.Columns.Count

    if max_row >= 2:
        source_range = ws_avg.Range(ws_avg.Cells(2, 1), ws_avg.Cells(max_row, max_col))
        source_range.Copy()
        ws_final.Range("A2").PasteSpecial(Paste=-4104)
        excel.Application.CutCopyMode = False

        ws_final.Range("D2:BM2").Replace("합계 : ", "")

    ws_final.Rows(1).RowHeight = 26.25
    ws_final.Rows(2).RowHeight = 181.5

    ws_final.Columns("A").ColumnWidth = 3.75
    ws_final.Columns("B").ColumnWidth = 12.75
    ws_final.Columns("C").ColumnWidth = 7.38

    for col in ["D", "AJ", "AU", "AV", "BK"]:
        ws_final.Columns(col).ColumnWidth = 5.5

    for col_range in ["E:AI", "AK:AT", "AW:BB", "BE:BJ"]:
        ws_final.Columns(col_range).ColumnWidth = 4

    ws_final.Columns("BC").ColumnWidth = 4.88
    ws_final.Columns("BD").ColumnWidth = 5.25
    ws_final.Range("BL:BM").ColumnWidth = 6.88

    ws_final.Rows(2).WrapText = True

    ws_final.Rows(2).Font.Name = "맑은 고딕"
    ws_final.Rows(2).Font.Size = 11

    if max_row >= 3:
        data_range = ws_final.Range(ws_final.Cells(3, 1), ws_final.Cells(max_row, max_col))
        data_range.Font.Name = "WenQuanYi Zen Hei"
        data_range.Font.Size = 10

    ws_final.Range("BL:BM").EntireColumn.Hidden = True

    if prefix in ['KR', 'US']:
        fill_color = 219 + (238 * 256) + (243 * 65536)
    elif prefix == 'NK':
        fill_color = 243 + (220 * 256) + (219 * 65536)

    if max_row >= 2:
        for r in range(2, max_row + 1):
            for c in range(1, max_col + 1):
                val = ws_final.Cells(r, c).Value
                if type(val) in (int, float, str) and val != 0:
                    ws_final.Cells(r, c).Interior.Color = fill_color

        border_range = ws_final.Range(ws_final.Cells(2, 1), ws_final.Cells(max_row, max_col))
        border_color = 150 + (179 * 256) + (215 * 65536)

        for border_id in [7, 8, 9, 10, 11, 12]:
            border_range.Borders(border_id).LineStyle = 1
            border_range.Borders(border_id).Color = border_color

    print(f"{ko_name} 최종 시트 생성")
    
def insert_dat_to_excel(wb, folder, file_name, sheet_name):
    path = os.path.join(folder, file_name)
    if not os.path.exists(path):
        print(f"{file_name} 파일이 없음")
        return
    
    ws = wb.Sheets.Add(After=wb.Sheets(wb.Sheets.Count))
    ws.Name = sheet_name
    
    try:
        with open(path, 'r', encoding='utf-8-sig') as f:
            lines = f.readlines()
    except UnicodeDecodeError:
        with open(path, 'r', encoding='cp949') as f:
            lines = f.readlines()
            
    data = [line.rstrip('\n').split('\t') for line in lines]
    
    if data:
        row_count = len(data)
        col_count = max(len(r) for r in data)
        
        for r in data:
            r.extend([""] * (col_count - len(r)))
            
        ws.Range(ws.Cells(1, 1), ws.Cells(row_count, col_count)).Value = data
        print(f"{sheet_name} 시트 생성 데이터 복사 완료")

In [7]:
for x in ['KR','US','NK','kill']:
    if x == 'kill':
        try:
            FirstStepOfKill()
            time.sleep(3)
            SecondStepOfKill()
        except:
            pass
    else:
        try:
            FirstStepOfSortie(x)
            time.sleep(3)
            SecondStepOfSortie(x)
        except:
            pass

time.sleep(3)
folder_path = r"C:\Users\LAB1\Desktop\대기실" # ◉◉◉◉바꿀 수 있는 위치(위 함수 모음이랑 같아야 함)◉◉◉◉
wb = Workbook()
wb.remove(wb.active)


for alpha in ['합친 것_KR','합친 것_US','합친 것_NK','합친 것_kill']:
    try:
        csv_path = os.path.join(folder_path, f"{alpha}.csv")

        if not os.path.exists(csv_path):
            continue
        
        if alpha == '합친 것_KR':
            ws = wb.create_sheet(title='KR 나열')
        elif alpha == '합친 것_US':
            ws = wb.create_sheet(title='US 나열')
        elif alpha == '합친 것_NK':
            ws = wb.create_sheet(title='NK 나열')
        elif alpha == '합친 것_kill':
            ws = wb.create_sheet(title='A2S kill 나열')

        with open(csv_path, mode="r", encoding="utf-8-sig", newline="") as f:
            reader = csv.reader(f)

            for _ in range(3):
                next(reader, None)

            for row_idx, row in enumerate(reader, start=1):
                for col_idx, value in enumerate(row, start=1):
                    ws.cell(
                        row=row_idx,
                        column=col_idx,
                        value=convert_value(value)
                    )
    except:
        pass
                
xlsx_path = os.path.join(folder_path, "Result.xlsx")
wb.save(xlsx_path)
wb.close()

print("1단계 완료")

time.sleep(3)

try:
    excel = win32.Dispatch('Excel.Application')
    excel.Visible = False
    excel.DisplayAlerts = False
    
    abs_xlsx_path = os.path.abspath(xlsx_path)
    wb_pv = excel.Workbooks.Open(abs_xlsx_path)
        
    create_static_pivot('KR 나열', 'KR_PV')
    create_static_pivot('US 나열', 'US_PV')
    create_static_pivot('NK 나열', 'NK_PV')
    create_dynamic_pivot()
    
    create_avg_pivot('KR')
    create_avg_pivot('US')
    create_avg_pivot('NK')
        
    create_final_tab('KR', '한')
    create_final_tab('US', '미')
    create_final_tab('NK', '북')
    
    insert_dat_to_excel(wb_pv, folder_path, "CNO.dat", "CNO")
    insert_dat_to_excel(wb_pv, folder_path, "SA235.dat", "SA235")
    
    wb_pv.Save()
    wb_pv.Close()
    
except Exception as e:
    print(f"피벗 테이블 생성 중 오류 : {e}")

finally:
    try:
        excel.Quit()
    except:
        pass
    print("2단계 완료")

✅ '합친 것.csv' 파일이 성공적으로 생성되었습니다!
저장 위치: C:\Users\LAB1\Desktop\대기실\합친 것_KR.csv
✅ '합친 것.csv' 파일이 성공적으로 생성되었습니다!
저장 위치: C:\Users\LAB1\Desktop\대기실\합친 것_US.csv
✅ '합친 것.csv' 파일이 성공적으로 생성되었습니다!
저장 위치: C:\Users\LAB1\Desktop\대기실\합친 것_NK.csv
✅ '합친 것.csv' 파일이 성공적으로 생성되었습니다!
저장 위치: C:\Users\LAB1\Desktop\대기실\합친 것_kill.csv
1단계 완료
KR_PV 시트 생성
US_PV 시트 생성
NK_PV 시트 생성
A2S kill Pivot 시트 생성
KR_PV_AVG 시트 생성
US_PV_AVG 시트 생성
NK_PV_AVG 시트 생성
한 최종 시트 생성
미 최종 시트 생성
북 최종 시트 생성
CNO 시트 생성 데이터 복사 완료
SA235 시트 생성 데이터 복사 완료
2단계 완료
